## Verification of 2D Axisymmetric (RZ / `bidim_axi`) Implementation

This notebook aims to verify the 2D axisymmetric implementation by comparing **VDF** (validated reference) against **EF** and **PolyMAC_MPFA** discretizations.

### Scope
- **Physics:** solid heat conduction and incompressible hydraulics  
- **Geometries:** annular domain and full cylindrical domain (including **r = 0**)

### Methodology
The notebook runs identical cases for each discretization and compares radial profiles vs VDF.

In [ ]:
from trustutils import run
run.TRUST_parameters()
run.reset()
dis = {"VDF" : "-", "PolyMAC_MPFA" : "x", "EF" : "+"}
cases = ["thsol_a", "hyd_a", "thsol", "hyd"]
nb_part = 1
par = "" if nb_part == 1 else "PAR_"
for d in dis:
    for c in cases:
        tc = run.addCaseFromTemplate(f"jdd_{c}.data", f"{d}/{c}", {"dis": d, "conv" : "generic amont" if d == "EF" else "amont"}, nbProcs=nb_part)
        if (nb_part > 1): tc.partition()
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils import plot

for c in cases:
    a = plot.Graph(c)
    for d in dis:
        a.addResidu(f"{run.BUILD_DIRECTORY}/{d}/{c}/{par}jdd_{c}.dt_ev", label=d)

    a.scale(yscale="log")

In [ ]:
for c in cases:
    vars = ["T1", "T2"] if c.startswith("thsol") else ["V"]
    compo = 0 if c.startswith("thsol") else 1
    a = plot.Graph(c)
    for d, marker in dis.items():
        for var in vars:
            a.addSegment(f"{run.BUILD_DIRECTORY}/{d}/{c}/{par}jdd_{c}_{var}.son", label=f"{d} - {var}", compo=compo, marker=marker)

    a.legend()
